In [1]:
cd ..

/Users/camila.cusicanqui/Documents/GitHub/frod-agentic-ai


In [2]:
%load_ext autoreload
%autoreload 2

In [36]:
import os
import yaml

with open(".config/credentials-mage.yaml", "r") as f:
    creds = yaml.safe_load(f)

for item in creds:
    os.environ[item["name"]] = str(item["value"])

In [19]:
import importlib.util
import sys
from pathlib import Path

TRANSACTION_SCHEMA_PATH = Path(
    "/Users/camila.cusicanqui/Documents/GitHub/analytics-mage-infra/mage-fraud-space/fraud_utils/fraud_agents/schemas/transactions.py"
)
spec = importlib.util.spec_from_file_location("fraud_transaction_schemas", TRANSACTION_SCHEMA_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load transaction schemas from {TRANSACTION_SCHEMA_PATH}")

transaction_schemas = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = transaction_schemas
spec.loader.exec_module(transaction_schemas)

TransactionRow = transaction_schemas.TransactionRow
TransactionBatch = transaction_schemas.TransactionBatch

In [34]:
# homemade utils
from utils.data_ingest import get_db_conn
from utils.chargeback_data import load_cb_df
# librerías 
import pandas as pd
import numpy as np
import gspread
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta, datetime
import gspread
from dateutil.utils import today

# formatting options
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format


In [6]:
cb_df = load_cb_df()

/Users/camila.cusicanqui/Documents/GitHub/frod-agentic-ai/utils/chargeback_data.py:201: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [11]:
cb_df["amount_pos"] = cb_df["amount"]*-1


In [ ]:
important_cols = [
    'cb_timestamp',
    'user_id',
    'transaction_id',
    'trx_timestamp_mx',
    'amount_pos',
    # 'merchant',
    'operador',
    'mcc_code',
    'card_type',
    'product_type',
    'nombrearchivo',
    'embozo_file_date',
    'pos_entry_mode',
    'cvv_ind',
    'metodo_identificacion',
    'three_ds_flow',
    'afiliacion',
    'adquirente',
    'country',
    'cod_respuesta',   
]

In [43]:
cb_df

,freshdesk_ticket_id,cb_reason,cb_timestamp,user_id,klrid,transaction_id,trx_timestamp_mx,amount,merchant,mcc_code,card_type,product_type,nombrearchivo,pos_entry_mode,cvv_ind,t3ds_indicator,cb_source,sync_timestamp_utc,omi_id_operacion_raw,metodo_identificacion,leading_indicator,three_ds_status,three_ds_flow,c063,afiliacion,numero_terminal,adquirente,operador,cod_respuesta,trx_time,week_number,trx_date,trx_month,trx_day,trx_month_year,trx_week_year,cb_date,cb_month,cb_day,cb_month_year,cb_week_year,online,cb_reason_original,country,amount_pos,embozo_file_date
0,12544637,Otro,2025-06-03 22:44:13,44f3688e-45fd-44ea-81fe-b22d8c7cf1d8,052f20dd-dae2-41c6-b070-f19bda9466aa,PARABILIUM:100159166,2025-05-18 21:57:55.570,-800.00,OPENPAYADMA PROVIDENCIA 1JALMX,7277,VIRTUAL,CREDIT_5401_BIN,NaN,CNP Manual,has cvv,3ds,freshdesk,2025-12-09 23:10:45,100159166,NaN,kB,3DS_AUTHENTICATED,CHALLENGE,& 0000600344! C000026 XXXXXXXXXXXXXXXXXXXXXXXX...,4200578,03040378,12,OPENPAY*ADMA PROVIDENCIA 1JALMX,00,2025-05-18 21:57:55.570,20,2025-05-18,5.00,18.00,2025-05,2025-19,2025-06-03,6,3,2025-06,2025-22,True,Otro,MX,800.00,NaT
1,12401525,Otro,2025-05-21 15:12:43,18918cbe-d254-425a-8a11-9b7cf98d8b4d,942962f2-29c2-4ee1-ac59-1f2be7a21f32,PARABILIUM:100463184,2025-05-19 22:28:42.643,"-2,774.00",VOLARIS MOTO DEB 2 CIUDAD DE MEX001MX,4511,PHYSICAL,PLATINUM,par250125ProdJSP02_SubEm0004.emb,CNP Manual,has cvv,normal,freshdesk,2025-12-09 23:10:45,100463184,NaN,NaN,UNKNOWN,UNKNOWN,& 0000400082! C000026 XXXXXXXXXXXXXXXXXXXXXXXX...,8112910,208624789,5,VOLARIS MOTO DEB 2 CIUDAD DE MEX001MX,00,2025-05-19 22:28:42.643,21,2025-05-19,5.00,19.00,2025-05,2025-20,2025-05-21,5,21,2025-05,2025-20,True,Otro,MX,"2,774.00",2025-01-25
2,12595415,Otro,2025-06-09 01:47:29,e0b3bdb2-ec5f-4784-a5d4-9950225960c9,f768e9ac-1842-410f-b068-970e5743314b,PARABILIUM:100679780,2025-05-20 18:30:51.097,-109.00,TRACTEBEL DGQRO MU MEXICO DF DF MX,5983,PHYSICAL,CREDIT_5401_BIN,par240403ProdJSO01_SubEm0001.emb,CNP Manual,has cvv,3ds,freshdesk,2025-12-09 23:10:45,100679780,NaN,kA,3DS_AUTHENTICATED,FRICTIONLESS,& 0000600344! C000026 XXXXXXXXXXXXXXXXXXXXXXXX...,4040922,00007319,12,TRACTEBEL DGQRO MU MEXICO DF DF MX,00,2025-05-20 18:30:51.097,21,2025-05-20,5.00,20.00,2025-05,2025-20,2025-06-09,6,9,2025-06,2025-23,True,Otro,MX,109.00,2024-04-03
3,12460106,Cargo no reconocido,2025-05-26 20:50:10,f8fb6af7-bb15-40e2-a06a-3206ff35a40f,cfd80a63-7763-428a-8323-f2cdf3565d2e,101015177,2025-05-21 20:01:21.900,-349.00,KFC ECOMMERCE CIUDAD DE MEX001MX,5814,VIRTUAL,CREDIT_5401_BIN,NaN,CNP Manual,has cvv,3ds,freshdesk,2025-12-09 23:10:45,101015177,NaN,kB,3DS_AUTHENTICATED,CHALLENGE,& 0000600344! C000026 XXXXXXXXXXXXXXXXXXXXXXXX...,8419607,06699918,302,KFC ECOMMERCE CIUDAD DE MEX001MX,00,2025-05-21 20:01:21.900,21,2025-05-21,5.00,21.00,2025-05,2025-20,2025-05-26,5,26,2025-05,2025-21,True,Error durante el proceso,MX,349.00,NaT
4,12478081,Otro,2025-05-28 14:26:40,69cc717e-f27b-4a0f-b46f-3315d16fd228,07d988fc-3bab-441e-b6fe-1facab01267d,PARABILIUM:101804609,2025-05-24 13:16:47.739,-263.00,SERV CONSULTORIA MK HUIXQUILUCAN 015MX,7392,VIRTUAL,CREDIT_5456_BIN,NaN,CNP Manual,no cvv,normal,freshdesk,2025-12-09 23:10:45,101804609,NaN,NaN,UNKNOWN,UNKNOWN,& 0000500112! C000026 XXXXXXXXXXXXXXXXXXXXXXXX...,9628274,0000000000000001,485,SERV CONSULTORIA MK HUIXQUILUCAN 015MX,00,2025-05-24 13:16:47.739,21,2025-05-24,5.00,24.00,2025-05,2025-20,2025-05-28,5,28,2025-05,2025-21,True,Otro,MX,263.00,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273751,14210678,Otro,2025-10-24 22:28:43,b69e2053-e9de-4cf7-af2f-350209fa7e8b,c09c3cd9-1670-4fbd-8010-29bfe1917543,c09c3cd9-1670-4fbd-8010-29bfe1917543:388852:SI...,2025-10-21 18:33:24.832,-149.00,,0731,PHYSICAL,PLATINUM,par250812ProdJSP01_SubEm0004.emb,NaN,unknown,normal,freshdesk,2025-12-09 23:10:45,c09c3cd9-1670-4fbd-8010-29bfe1917543:388852:SI...,NaN,NaN

In [17]:
cb_df[important_cols]

,cb_timestamp,user_id,transaction_id,trx_timestamp_mx,amount_pos,operador,mcc_code,card_type,product_type,nombrearchivo,pos_entry_mode,cvv_ind,metodo_identificacion,three_ds_flow,afiliacion,adquirente,country,cod_respuesta
0,2025-06-03 22:44:13,44f3688e-45fd-44ea-81fe-b22d8c7cf1d8,PARABILIUM:100159166,2025-05-18 21:57:55.570,800.00,OPENPAY*ADMA PROVIDENCIA 1JALMX,7277,VIRTUAL,CREDIT_5401_BIN,NaN,CNP Manual,has cvv,NaN,CHALLENGE,4200578,12,MX,00
1,2025-05-21 15:12:43,18918cbe-d254-425a-8a11-9b7cf98d8b4d,PARABILIUM:100463184,2025-05-19 22:28:42.643,"2,774.00",VOLARIS MOTO DEB 2 CIUDAD DE MEX001MX,4511,PHYSICAL,PLATINUM,par250125ProdJSP02_SubEm0004.emb,CNP Manual,has cvv,NaN,UNKNOWN,8112910,5,MX,00
2,2025-06-09 01:47:29,e0b3bdb2-ec5f-4784-a5d4-9950225960c9,PARABILIUM:100679780,2025-05-20 18:30:51.097,109.00,TRACTEBEL DGQRO MU MEXICO DF DF MX,5983,PHYSICAL,CREDIT_5401_BIN,par240403ProdJSO01_SubEm0001.emb,CNP Manual,has cvv,NaN,FRICTIONLESS,4040922,12,MX,00
3,2025-05-26 20:50:10,f8fb6af7-bb15-40e2-a06a-3206ff35a40f,101015177,2025-05-21 20:01:21.900,349.00,KFC ECOMMERCE CIUDAD DE MEX001MX,5814,VIRTUAL,CREDIT_5401_BIN,NaN,CNP Manual,has cvv,NaN,CHALLENGE,8419607,302,MX,00
4,2025-05-28 14:26:40,69cc717e-f27b-4a0f-b46f-3315d16fd228,PARABILIUM:101804609,2025-05-24 13:16:47.739,263.00,SERV CONSULTORIA MK HUIXQUILUCAN 015MX,7392,VIRTUAL,CREDIT_5456_BIN,NaN,CNP Manual,no cvv,NaN,UNKNOWN,9628274,485,MX,00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273751,2025-10-24 22:28:43,b69e2053-e9de-4cf7-af2f-350209fa7e8b,c09c3cd9-1670-4fbd-8010-29bfe1917543:388852:SI...,2025-10-21 18:33:24.832,149.00,NaN,0731,PHYSICAL,PLATINUM,par250812ProdJSP01_SubEm0004.emb,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN
273752,2025-09-24 23:10:48,3d2a5350-fa17-4a11-9d00-1e08e2431976,dd56284b-bf95-4912-ab11-083c659fb081:970959:88...,2025-09-24 16:02:31.436,367.53,NaN,0412,PHYSICAL,CREDIT_5401_BIN,par250916ProdJSO02_SubEm0001.emb,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN
273753,2025-03-21 20:16:49,14a6bb69-4303-4e85-900a-67a1c8425ce3,ed4a9f04-6c8b-4a7c-9aac-4cda56b42e15: :00...,2025-03-05 17:59:41.078,721.50,NaN,0507,PHYSICAL,CREDIT_5401_BIN,par241124ProdJSO01_SubEm0001.emb,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN
273754,2025-09-09 12:35:59,cba1d2c0-6bdd-42ad-b753-e6ae94fcffae,f60976f9-0391-404d-a390-42ea83e3ecc5:830569:3O...,2025-09-08 18:01:16.906,99.00,NaN,0489,PHYSICAL,CREDIT_5456_BIN,NaN,NaN,unknown,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
cb_df.groupby(['three_ds_status','three_ds_flow']).transaction_id.nunique()

three_ds_status        three_ds_flow 
3DS_AUTHENTICATED      CHALLENGE          10993
                       FRICTIONLESS        2987
3DS_NOT_AUTHENTICATED  EXEMPT_OR_INFO        52
UNKNOWN                UNKNOWN           241569
Name: transaction_id, dtype: int64

In [35]:
import asyncio
#from agents import Agent, Runner
#from agents import Agent, Runner, function_tool
import datetime as dt
import polars as pl
# import docx2txt
from pydantic import BaseModel, Field
from typing import List, Optional
import json


In [25]:
def _transaction_value_for_prompt(column, value):
    if pd.isna(value):
        return None
    if column == "amount_pos":
        return float(value)
    if hasattr(value, "isoformat"):
        return value.isoformat(sep=" ")
    return str(value)


def build_transaction_batch(df: pd.DataFrame, limit: int | None = None) -> TransactionBatch:
    rows_df = df.loc[:, important_cols]
    if limit is not None:
        rows_df = rows_df.head(limit)

    records = [
        {
            column: _transaction_value_for_prompt(column, value)
            for column, value in row.items()
        }
        for row in rows_df.to_dict("records")
    ]
    return TransactionBatch(transactions=records)



In [26]:

transactions_for_prompt = build_transaction_batch(cb_df, limit=50)
transaction_context = transactions_for_prompt.model_dump_json(indent=2)
user_prompt = f"""Describe the transactional patterns in this chargeback batch.

Transactions:
{transaction_context}
"""

user_prompt

'Describe the transactional patterns in this chargeback batch.\n\nTransactions:\n{\n  "transactions": [\n    {\n      "cb_timestamp": "2025-06-03 22:44:13",\n      "user_id": "44f3688e-45fd-44ea-81fe-b22d8c7cf1d8",\n      "transaction_id": "PARABILIUM:100159166",\n      "trx_timestamp_mx": "2025-05-18 21:57:55.570000",\n      "amount_pos": 800.0,\n      "operador": "OPENPAY*ADMA          PROVIDENCIA 1JALMX",\n      "mcc_code": "7277",\n      "card_type": "VIRTUAL",\n      "product_type": "CREDIT_5401_BIN",\n      "nombrearchivo": null,\n      "pos_entry_mode": "CNP Manual",\n      "cvv_ind": "has cvv",\n      "metodo_identificacion": null,\n      "three_ds_flow": "CHALLENGE",\n      "afiliacion": "4200578",\n      "adquirente": "12",\n      "country": "MX",\n      "cod_respuesta": "00  "\n    },\n    {\n      "cb_timestamp": "2025-05-21 15:12:43",\n      "user_id": "18918cbe-d254-425a-8a11-9b7cf98d8b4d",\n      "transaction_id": "PARABILIUM:100463184",\n      "trx_timestamp_mx": "2025-

In [27]:
transaction_context

'{\n  "transactions": [\n    {\n      "cb_timestamp": "2025-06-03 22:44:13",\n      "user_id": "44f3688e-45fd-44ea-81fe-b22d8c7cf1d8",\n      "transaction_id": "PARABILIUM:100159166",\n      "trx_timestamp_mx": "2025-05-18 21:57:55.570000",\n      "amount_pos": 800.0,\n      "operador": "OPENPAY*ADMA          PROVIDENCIA 1JALMX",\n      "mcc_code": "7277",\n      "card_type": "VIRTUAL",\n      "product_type": "CREDIT_5401_BIN",\n      "nombrearchivo": null,\n      "pos_entry_mode": "CNP Manual",\n      "cvv_ind": "has cvv",\n      "metodo_identificacion": null,\n      "three_ds_flow": "CHALLENGE",\n      "afiliacion": "4200578",\n      "adquirente": "12",\n      "country": "MX",\n      "cod_respuesta": "00  "\n    },\n    {\n      "cb_timestamp": "2025-05-21 15:12:43",\n      "user_id": "18918cbe-d254-425a-8a11-9b7cf98d8b4d",\n      "transaction_id": "PARABILIUM:100463184",\n      "trx_timestamp_mx": "2025-05-19 22:28:42.643000",\n      "amount_pos": 2774.0,\n      "operador": "VOLARIS

## Local Python transaction analyst agent

These cells expose `cb_df[important_cols]` to the agent through controlled local Python tools instead of dumping the full dataframe into the prompt.

In [28]:
from types import SimpleNamespace

from transactional_describer.transaction_agent import (
    TransactionalDescription,
    build_transaction_analyst_agent,
    get_transaction_schema,
    prepare_transaction_context,
    run_transaction_python,
)

In [29]:
transaction_agent_context = prepare_transaction_context(
    cb_df[cb_df['trx_timestamp_mx'] >= '2026-05-25'],
    important_cols,
    sample_name="chargeback_transactions",
)

print(f"Saved agent dataset to: {transaction_agent_context.data_path}")
print(f"Columns exposed: {transaction_agent_context.columns}")

Saved agent dataset to: /var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/chargeback_transactions_9c83fd49fe154846a56dbf38414a4fb0.jsonl
Columns exposed: ['cb_timestamp', 'user_id', 'transaction_id', 'trx_timestamp_mx', 'amount_pos', 'operador', 'mcc_code', 'card_type', 'product_type', 'nombrearchivo', 'pos_entry_mode', 'cvv_ind', 'metodo_identificacion', 'three_ds_flow', 'afiliacion', 'adquirente', 'country', 'cod_respuesta']


In [30]:
schema_preview = get_transaction_schema(SimpleNamespace(context=transaction_agent_context))
print(schema_preview[:4000])

{"row_count": 1486, "column_count": 18, "columns_exposed": ["cb_timestamp", "user_id", "transaction_id", "trx_timestamp_mx", "amount_pos", "operador", "mcc_code", "card_type", "product_type", "nombrearchivo", "pos_entry_mode", "cvv_ind", "metodo_identificacion", "three_ds_flow", "afiliacion", "adquirente", "country", "cod_respuesta"], "columns": [{"name": "cb_timestamp", "dtype": "str", "null_count": 0, "null_rate": 0.0, "sample_values": ["2026-05-29T01:23:46.539", "2026-05-30T18:42:44.344", "2026-05-30T13:35:42.202", "2026-05-29T06:15:37.986", "2026-05-29T02:36:39.048"]}, {"name": "user_id", "dtype": "str", "null_count": 0, "null_rate": 0.0, "sample_values": ["38248b6b-82d0-4d7b-a5b5-24cb53934dc7", "568eacf2-e234-4e76-a0fe-e754384f27e1", "ca066a4c-6c3e-472b-8cdd-d070e35947ed", "1b7ab48f-cf5e-4e61-8dda-9c3d41739b45", "135575d6-9e0b-4885-97b7-8f7da8ed5841"]}, {"name": "transaction_id", "dtype": "str", "null_count": 0, "null_rate": 0.0, "sample_values": ["PARABILIUM:235790753", "PARABILI

In [31]:
tool_preview = run_transaction_python(
    SimpleNamespace(context=transaction_agent_context),
    """
top_merchants = (
    df.groupby("operador", dropna=False)
      .agg(
          transactions=("transaction_id", "nunique"),
          users=("user_id", "nunique"),
          amount=("amount_pos", "sum"),
      )
      .sort_values(["transactions", "amount"], ascending=False)
      .head(10)
)
top_merchants
    """,
)
print(tool_preview[:5000])

{"status": "ok", "stdout": "", "result": {"type": "DataFrame", "shape": [10, 3], "columns": ["transactions", "users", "amount"], "records": [{"transactions": 169, "users": 144, "amount": 179507.54}, {"transactions": 69, "users": 62, "amount": 6550.21}, {"transactions": 50, "users": 44, "amount": 15388.6}, {"transactions": 47, "users": 30, "amount": 4530.0}, {"transactions": 37, "users": 31, "amount": 101889.03}, {"transactions": 26, "users": 25, "amount": 3122.0}, {"transactions": 24, "users": 24, "amount": 2515.0}, {"transactions": 20, "users": 20, "amount": 7769.81}, {"transactions": 18, "users": 15, "amount": 3632.58}, {"transactions": 16, "users": 11, "amount": 1450.0}]}}


In [37]:
from agents import Runner

transaction_agent = build_transaction_analyst_agent()

analyst_request = """
Describe the common transaction patterns in this chargeback batch.
Focus on common merchants, repeated user behavior, product/card type differences,
authentication patterns (CVV/3DS/POS entry), countries/acquirers, response codes, amounts, and timing.
Use Python before answering and cite the computed evidence.
""".strip()

transaction_agent_result = await Runner.run(
    transaction_agent,
    analyst_request,
    context=transaction_agent_context,
    max_turns=8,
)

transaction_description = transaction_agent_result.final_output
transaction_description.model_dump()

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


{'summary': 'This batch is dominated by low-value CNP activity, mostly in Mexico, with a small set of recurring merchants and many single-transaction users. Authentication data is sparse or defaulted in many records, so pattern confidence is strongest on merchant, country, amount, and entry-mode trends.',
 'key_patterns': [{'title': 'A few merchants repeat heavily, but most users are one-off',
   'evidence': 'Top merchants include DLO*UBER RIDE (69 tx), APPLE.COM/BILL (50), TELCEL RECARGAS FONY (47), MERCADO PAGO (37), and AMAZONCOM INC AMZN (26). User concentration is low: 1,146 users for 1,486 rows, with 170 users having >1 tx and max 15 tx/user.',
   'examples': []},
  {'title': 'CNP/manual transactions dominate',
   'evidence': 'pos_entry_mode shows CNP Manual 816/1,486 (54.9%), CNP Card On File 220 (14.8%), Contactless 171 (11.5%), and Integrated Circuit Read 110 (7.4%); 169 rows are null for POS entry mode.',
   'examples': []},
  {'title': 'CVV and 3DS are often missing/unknown'